In [2]:
import random
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

# 1. Deterministic Seeds & Device Configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active compute device: {device}")

# 1. DATA PIPELINE
print("\nLoading dataset 'papluca/language-identification'...")
raw_dataset = load_dataset("papluca/language-identification")

train_data = raw_dataset["train"]
val_data   = raw_dataset["validation"]
test_data  = raw_dataset["test"]

# Target classes mapping
languages = sorted(list(set(train_data["labels"])))
num_classes = len(languages)
label2id = {lang: idx for idx, lang in enumerate(languages)}
id2label = {idx: lang for idx, lang in enumerate(languages)}

print(f"Target Languages ({num_classes}): {languages}")
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

# Tokenization strictly following the reference notebook:
# Lowercase and split on whitespace / clean punctuation
def tokenize(text):
    return text.lower().replace("<br />", " ").split()

# Build vocabulary index dictionary over training split (Top 40,000)
word_counts = Counter()
for sample in train_data:
    word_counts.update(tokenize(sample["text"]))

VOCAB_CAP = 40000
top_words = [word for word, count in word_counts.most_common(VOCAB_CAP - 2)]

# Reserve special tokens: <pad> -> 0, <unk> -> 1
vocab = ['<pad>', '<unk>'] + top_words
word_to_id = {word: idx for idx, word in enumerate(vocab)}
id_to_word = {idx: word for idx, word in enumerate(vocab)}

PAD_IDX = word_to_id['<pad>']
UNK_IDX = word_to_id['<unk>']
VOCAB_SIZE = len(vocab)
MAX_LEN = 60  # Uniform sequence cutoff length

print(f"Vocabulary Size: {VOCAB_SIZE}")
print(f"Special Tokens: <pad> -> {PAD_IDX}, <unk> -> {UNK_IDX}")

# Standardize sequences via pre-padding
def encode_sequence(text, max_len=MAX_LEN):
    tokens = tokenize(text) if isinstance(text, str) else text
    ids = [word_to_id.get(token, UNK_IDX) for token in tokens[:max_len]]
    if len(ids) < max_len:
        ids = [PAD_IDX] * (max_len - len(ids)) + ids
    return ids

class LanguageDataset(Dataset):
    def __init__(self, hf_split):
        self.texts = hf_split["text"]
        self.labels = hf_split["labels"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = encode_sequence(self.texts[idx], MAX_LEN)
        y = label2id[self.labels[idx]]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

BATCH_SIZE = 128
train_loader = DataLoader(LanguageDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(LanguageDataset(val_data), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(LanguageDataset(test_data), batch_size=BATCH_SIZE, shuffle=False)

# 2. ARCHITECTURE DESIGN (LSTM Model)
class MultilingualLanguageLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=20, pad_idx=0, dropout=0.3):
        super(MultilingualLanguageLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embeds = self.embedding(x)                  # [batch, seq_len, embed_dim]
        out, (h_n, c_n) = self.lstm(embeds)         # out: [batch, seq_len, hidden_dim]
        last_step = out[:, -1, :]                   # [batch, hidden_dim]
        logits = self.fc(self.dropout(last_step))   # [batch, num_classes]
        return logits

model = MultilingualLanguageLSTM(
    vocab_size=VOCAB_SIZE,
    embed_dim=128,
    hidden_dim=128,
    num_classes=num_classes,
    pad_idx=PAD_IDX,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

print("\nModel Architecture:\n", model)

# 3. MODEL TRAINING & EVALUATION
def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct_preds = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = criterion(logits, targets)

            total_loss += loss.item() * targets.size(0)
            preds = logits.argmax(dim=1)
            correct_preds += preds.eq(targets).sum().item()
            total_samples += targets.size(0)

    avg_loss = total_loss / total_samples
    accuracy = (correct_preds / total_samples) * 100.0
    return avg_loss, accuracy

EPOCHS = 5
print("\nStarting Training Pipeline...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()

        # Gradient clipping to maintain numerical stability during BPTT (from Lecture)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        running_loss += loss.item() * targets.size(0)
        preds = logits.argmax(dim=1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)

    train_loss = running_loss / total
    train_acc = (correct / total) * 100.0
    val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)

    print(f"Epoch [{epoch}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%")

# Final Evaluation on the 10,000 Unseen Test Samples
test_loss, test_acc = evaluate_model(model, test_loader, criterion, device)
print(f"\n==========================================")
print(f"FINAL TEST EVALUATION (10,000 Test Samples):")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"==========================================")

# 4. CUSTOM INFERENCE PIPELINE
def predict_language(raw_text, model, top_k=5):
    """
    Accepts arbitrary customer message, tokenizes and pads,
    and returns predicted language with confidence probabilities.
    """
    model.eval()
    with torch.no_grad():
        encoded = encode_sequence(raw_text, max_len=MAX_LEN)
        tensor_in = torch.tensor([encoded], dtype=torch.long).to(device)
        logits = model(tensor_in)
        probabilities = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

    pred_id = int(np.argmax(probabilities))
    pred_lang = id2label[pred_id]
    confidence = probabilities[pred_id]

    top_indices = np.argsort(probabilities)[::-1][:top_k]
    top_predictions = [(id2label[idx], float(probabilities[idx])) for idx in top_indices]

    return pred_lang, confidence, top_predictions

# Production test queries
test_queries = [
    "Bonjour, j'ai une question concernant mon virement bancaire.",
    "مرحباً، أريد الاستفسار عن كشف الحساب البنكي الخاص بي.",
    "Hallo, ich möchte mein Bankkonto überprüfen.",
    "Hello, I need assistance with a cross-border transaction."
]

print("\n--- Standalone Production Inference Verification ---")
for query in test_queries:
    lang, conf, top_preds = predict_language(query, model)
    print(f"Text: '{query}'")
    print(f"Predicted: {lang.upper()} | Confidence: {conf*100:.2f}%")
    print(f"Top Probabilities: {top_preds}\n")

Active compute device: cpu

Loading dataset 'papluca/language-identification'...
Target Languages (20): ['ar', 'bg', 'de', 'el', 'en', 'es', 'fr', 'hi', 'it', 'ja', 'nl', 'pl', 'pt', 'ru', 'sw', 'th', 'tr', 'ur', 'vi', 'zh']
Train: 70000 | Val: 10000 | Test: 10000
Vocabulary Size: 40000
Special Tokens: <pad> -> 0, <unk> -> 1

Model Architecture:
 MultilingualLanguageLSTM(
  (embedding): Embedding(40000, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=20, bias=True)
)

Starting Training Pipeline...
Epoch [1/5] | Train Loss: 0.4754 - Train Acc: 85.16% | Val Loss: 0.6311 - Val Acc: 83.68%
Epoch [2/5] | Train Loss: 0.1344 - Train Acc: 95.06% | Val Loss: 0.5634 - Val Acc: 87.68%
Epoch [3/5] | Train Loss: 0.0983 - Train Acc: 96.03% | Val Loss: 0.5270 - Val Acc: 88.14%
Epoch [4/5] | Train Loss: 0.0860 - Train Acc: 96.36% | Val Loss: 0.5910 - Val Acc: 87.76%
Epoch [5/5] | Train Loss: 0.0803 -